In [0]:
from delta.tables import DeltaTable

dfExternalSales = spark.table("samples.bakehouse.sales_customers")

# Check if target table exists
if spark.catalog.tableExists("bronze.bronze_sales_customers"):
    # Perform merge (upsert): update if exists, insert if not
    deltaTable = DeltaTable.forName(spark, "bronze.bronze_sales_customers")
    deltaTable.alias("target").merge(
        dfExternalSales.alias("source"),
        "target.customerID = source.customerID"  # Match on customerID
    ).whenMatchedUpdateAll(
    ).whenNotMatchedInsertAll(
    ).execute()
else:
    # First time: create the table
    dfExternalSales.write.saveAsTable("bronze.bronze_sales_customers")